# Migrate from MLflow to LangSmith

Choose where you will run and continue developing your application. Both paths migrate the same source resources and use LangSmith for future observability. Historical migration and changing application instrumentation are separate steps.

## Contents

1. [Stay on Databricks, switch observability to LangSmith](#scenario-1)
2. [Move development locally, use LangSmith](#scenario-2)
3. [Configure your chosen path](#configure)
4. [Export existing MLflow resources](#export)
5. [Preview the migration](#preview)
6. [Import into LangSmith and validate](#import)
7. [Send new application traces to LangSmith](#continue)

Select an experiment ID to discover linked datasets, prompts, traces, assessments, and run metadata without a setup manifest. An optional manifest route retains the richer setup-specific evaluation mappings. It does not move compute, deploy the application, or automatically translate every MLflow feature.

<a id="scenario-1"></a>
## 1. Stay on Databricks, switch observability to LangSmith

Keep your agent code and compute in Databricks. Run this notebook there to export MLflow history, import supported resources into LangSmith, and validate the result. Then instrument your Databricks application with LangSmith for new traces.

1. Import this notebook and attach your existing compute. Use an environment with `mlflow`, the Databricks integrations, `langsmith`, and `langchain-core` for prompt import.
2. Leave the configuration on automatic environment detection and live-source export. Select the existing experiment ID from the discovery cell. Source access uses notebook authentication; the exporter reads the SQL warehouse ID from the experiment monitoring tag.
3. Configure LangSmith access through your workspace's secret-backed environment before the import step. Set the destination workspace explicitly. No secret values belong in saved cells.
4. Run the shared export and preview steps. Enable import only after reviewing the plan, then validate and run the opt-in LangSmith tracing check.

Your agent continues running on Databricks. LangSmith receives new telemetry; the historical MLflow resources remain available.

<a id="scenario-2"></a>
## 2. Move development locally, use LangSmith

Run the migration and future application development from a local Python environment or Jupyter notebook. LangSmith remains the destination observability service; running locally does not mean self-hosting LangSmith.

Choose one source route:

| Route | What to bring locally | Source access needed locally |
| --- | --- | --- |
| Export directly from Databricks | Use `EXPORT_SOURCE = True` and select an experiment ID; no manifest download | A Databricks SDK profile or configured authentication with access to the experiment, UC resources, artifacts, and SQL warehouse |
| Import an already exported bundle | Export once on Databricks; download `bundle.json` to `./migration-export/bundle.json`; use `EXPORT_SOURCE = False` | None; import reads the local bundle |

For local direct export, authenticate once with the Databricks CLI:

```bash
databricks auth login --host https://YOUR-WORKSPACE.cloud.databricks.com --profile mlflow-migration
```

Then set `MLFLOW_TRACKING_URI` to `databricks://YOUR_PROFILE` in your local environment, or fill SOURCE_TRACKING_URI below. The exporter fetches linked resources directly; no setup notebook is required. From a Databricks notebook, the same scripts use inherited authentication. From a repository checkout, the equivalent CLI flow is:

```bash
python providers/mlflow/main.py --list-experiments --tracking-uri databricks://mlflow-migration
python providers/mlflow/main.py --experiment EXPERIMENT_ID --tracking-uri databricks://mlflow-migration --output migration-export
python providers/mlflow/main.py --bundle migration-export/bundle.json --output migration-export --apply
```

Use a local Python/Jupyter environment with `mlflow` and the Databricks integrations for live export, plus `langsmith` and `langchain-core` for destination import. Configure LangSmith access in the local environment. No packages are installed by this notebook.

After migration, bring your application code and its dependencies locally, replace Databricks-only tools or configure remote access to them, and instrument the application with LangSmith. Migrating historical records does not port that application code for you.

<a id="configure"></a>
## 3. Configure your chosen path

Run all with defaults for export and preview only. Environment detection chooses Databricks workspace paths or local paths; override EXECUTION_ENVIRONMENT if necessary. For the exported-bundle route, set EXPORT_SOURCE to False and point BUNDLE_PATH at the downloaded bundle. Keep the prefix and state file stable when resuming the same migration.

In [ ]:
from pathlib import Path
import json
import os

EXECUTION_ENVIRONMENT = "auto"  # "auto", "databricks", or "local"
EXPORT_SOURCE = True  # False imports an existing local bundle without source access.
SOURCE_EXPERIMENT_ID = os.getenv("MLFLOW_EXPERIMENT_ID")  # Select from the listing below.
USE_SETUP_MANIFEST = False  # Optional legacy route for setup-specific evaluation mappings.
SOURCE_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")  # Locally: "databricks://YOUR_PROFILE"
PREFIX = "mlflow-uc-stack"

if EXECUTION_ENVIRONMENT not in {"auto", "databricks", "local"}:
    raise ValueError("Choose auto, databricks, or local.")
in_databricks = (EXECUTION_ENVIRONMENT == "databricks" or
    (EXECUTION_ENVIRONMENT == "auto" and
     (bool(os.getenv("DATABRICKS_RUNTIME_VERSION")) or "dbutils" in globals())))

MANIFEST_PATH = Path(os.getenv("MLFLOW_SETUP_MANIFEST", "./source/manifest.json"))
EXPORT_DIR = Path(os.getenv("MLFLOW_MIGRATION_OUTPUT", "./migration-export"))
BUNDLE_PATH = EXPORT_DIR / "bundle.json"  # Change if your downloaded bundle is elsewhere.
if EXPORT_SOURCE and USE_SETUP_MANIFEST:
    if not MANIFEST_PATH.is_file():
        raise FileNotFoundError("Set MANIFEST_PATH to an existing inventory.")
    manifest = json.loads(MANIFEST_PATH.read_text())
    print("Source experiment:", manifest["experiment_name"])
elif not EXPORT_SOURCE and not BUNDLE_PATH.is_file():
    raise FileNotFoundError("Set BUNDLE_PATH to your downloaded bundle.json.")
print("Running on:", "Databricks" if in_databricks else "local")
print("Output directory:", EXPORT_DIR)

### Load the migration code

The next cell loads the embedded exporter, transformations, importer, and validator. It creates no source resources and installs no packages.

In [ ]:
import sys
import tempfile

MODULES = {'stack_common.py': '"""Portable schema and deterministic demo fixtures. No third-party dependencies."""\nfrom copy import deepcopy\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\nimport os\n\nSCHEMA_VERSION = 1\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n    os.replace(temporary, path)\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef plain(value):\n    """Normalize SDK outputs, rejecting unsupported values rather than stringifying them."""\n    if value is None or isinstance(value, (str, bool, int)):\n        return value\n    if isinstance(value, float):\n        return value if math.isfinite(value) else None\n    if isinstance(value, dict):\n        return {str(k): plain(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [plain(v) for v in value]\n    if hasattr(value, \'to_dictionary\'):\n        return plain(value.to_dictionary())\n    if hasattr(value, \'to_dict\'):\n        return plain(value.to_dict())\n    if hasattr(value, \'to_proto\'):\n        from google.protobuf.json_format import MessageToDict\n        return plain(MessageToDict(value.to_proto(), preserving_proto_field_name=True))\n    if hasattr(value, \'item\'):\n        return plain(value.item())\n    if hasattr(value, \'isoformat\'):\n        return value.isoformat()\n    raise TypeError(f\'Unsupported export type: {type(value).__name__}\')\n\n\ndef fixtures():\n    # Independently authored references. The agent implements its own factual logic.\n    scenarios = [\n        (\'delivery\', [\'Where is DEMO-101?\', \'Will it arrive before the weekend?\', \'Do I need to sign?\'],\n         [\'Your order is expected Friday.\', \'Yes, delivery is expected Friday.\', \'No signature is required.\'],\n         [[\'lookup_order\', \'track_shipment\'], [\'track_shipment\'], [\'track_shipment\']]),\n        (\'returns\', [\'Can I return headphones bought 12 days ago?\', \'Even with an opened box?\', \'Please start my return.\'],\n         [\'Yes, the return window is 30 days.\', \'Opened items are eligible.\', \'A simulated return ticket was created; no real action was taken.\'],\n         [[\'check_return_eligibility\', \'search_policy\'], [\'search_policy\'], [\'create_demo_ticket\']]),\n        (\'billing\', [\'Why are there two charges?\', \'Did I pay twice?\', \'Please escalate this.\'],\n         [\'One charge is posted and one is pending.\', \'Only one charge has posted.\', \'A simulated billing ticket was created; no real action was taken.\'],\n         [[\'lookup_payments\', \'search_policy\'], [\'lookup_payments\'], [\'create_demo_ticket\']]),\n        (\'retry\', [\'Track DEMO-202, please.\', \'Where is it now?\', \'Can someone follow up?\'],\n         [\'The retry succeeded; delivery is expected Friday.\', \'Your package is at the local depot.\', \'A simulated delivery ticket was created; no real action was taken.\'],\n         [[\'lookup_order\', \'track_shipment\', \'track_shipment\'], [\'track_shipment\'], [\'create_demo_ticket\']]),\n    ]\n    rows = []\n    for scenario, questions, answers, tools in scenarios:\n        history = [{\'role\': \'system\', \'content\': \'You are a fictional support assistant.\'}]\n        for turn, (question, expected, expected_tools) in enumerate(zip(questions, answers, tools)):\n            history.append({\'role\': \'user\', \'content\': question})\n            rows.append({\n                \'id\': f\'{scenario}-{turn}\',\n                \'inputs\': {\'case_id\': f\'{scenario}-{turn}\', \'messages\': deepcopy(history)},\n                \'expectations\': {\'answer\': expected, \'tools\': expected_tools,\n                                 \'escalated\': turn == 2 and scenario != \'delivery\'},\n                \'metadata\': {\'scenario\': scenario, \'turn\': turn, \'synthetic\': True},\n            })\n            history.append({\'role\': \'assistant\', \'content\': expected})\n    return rows\n\n\ndef score_answer(outputs, expectations):\n    return outputs[\'answer\'].strip().casefold() == expectations[\'answer\'].strip().casefold()\n\n\ndef score_tools(outputs, expectations):\n    # Order and repetition matter: the retry must call track_shipment twice.\n    return outputs[\'tools\'] == expectations[\'tools\']\n\n\ndef score_escalation(outputs, expectations):\n    return outputs[\'escalated\'] == expectations[\'escalated\']\n\n\nSCORERS = {\'answer_match\': score_answer, \'tool_sequence\': score_tools,\n           \'escalation_match\': score_escalation}\nPROMPTS = [\n    {\'version\': \'1\', \'template\': \'Answer this support question: {{question}}\', \'aliases\': [\'baseline\']},\n    {\'version\': \'2\', \'template\': \'Use verified demo facts. Retry transient failures and escalate when requested. Question: {{question}}\', \'aliases\': [\'production\']},\n]\n\n\ndef is_databricks_uri(uri):\n    return uri == \'databricks\' or uri.startswith(\'databricks://\')\n\n\ndef databricks_registry_uri(tracking_uri):\n    if not is_databricks_uri(tracking_uri):\n        raise ValueError(\'A Databricks tracking URI is required\')\n    return tracking_uri.replace(\'databricks\', \'databricks-uc\', 1)\n', 'export_mlflow.py': '"""Export only resources identified by a seed manifest; never scan a whole workspace."""\nfrom pathlib import Path\nimport argparse\nimport json\nimport os\nimport time\nfrom copy import deepcopy\nfrom stack_common import SCHEMA_VERSION, digest, plain, save_json, is_databricks_uri, databricks_registry_uri\n\n\ndef normalize_trace(trace, link):\n    info = trace.info\n    metadata = getattr(info, \'trace_metadata\', None)\n    if metadata is None:\n        metadata = getattr(info, \'request_metadata\', {})\n    spans = []\n    for span in trace.data.spans:\n        status = str(span.status.status_code).split(\'.\')[-1]\n        spans.append({\n            \'id\': span.span_id, \'parent_id\': span.parent_id, \'name\': span.name,\n            \'type\': span.span_type, \'start_ns\': span.start_time_ns, \'end_ns\': span.end_time_ns,\n            \'inputs\': plain(span.inputs), \'outputs\': plain(span.outputs),\n            \'status\': status, \'error\': span.status.description if status == \'ERROR\' else None,\n            \'attributes\': plain(span.attributes),\n        })\n    assessments = []\n    for a in info.assessments or []:\n        kind = \'feedback\' if a.feedback is not None else (\'expectation\' if a.expectation is not None else \'issue\')\n        assessments.append({\n            \'id\': a.assessment_id, \'kind\': kind, \'name\': a.name,\n            \'value\': plain(a.value) if kind in (\'feedback\', \'expectation\') else None,\n            \'span_id\': a.span_id, \'rationale\': a.rationale,\n            \'source\': plain(a.source), \'metadata\': plain(a.metadata),\n            \'valid\': a.valid, \'overrides\': a.overrides,\n            \'error\': plain(a.feedback.error) if kind == \'feedback\' and a.feedback.error else None,\n            \'raw\': plain(a),\n        })\n    return {\'id\': info.trace_id, \'metadata\': plain(metadata), \'tags\': plain(info.tags),\n            \'session_id\': (metadata or {}).get(\'mlflow.trace.session\') or link.get(\'session_id\'),\n            \'link\': deepcopy(link), \'spans\': spans, \'assessments\': assessments,\n            \'raw\': plain(trace)}\n\n\ndef reconcile_evaluation_links(mlflow, manifest):\n    """Prediction callbacks may run outside the driver; persisted runs are authoritative."""\n    links = {entry[\'trace_id\']: deepcopy(entry) for entry in manifest[\'traces\']}\n    expected_cases = {row[\'id\'] for row in manifest[\'dataset\'][\'records\']}\n    reports, errors = [], []\n    for evaluation in manifest[\'evaluations\']:\n        if evaluation.get(\'status\') != \'created\' or not evaluation.get(\'run_id\'):\n            continue\n        try:\n            traces = mlflow.search_traces(\n                locations=[manifest[\'experiment_id\']], run_id=evaluation[\'run_id\'],\n                return_type=\'list\', max_results=10000,\n            )\n            if len(traces) >= 10000:\n                raise ValueError(\'Evaluation trace search reached its safety bound; pagination required\')\n            found = {}\n            seen_cases = set()\n            for trace in traces:\n                candidates = set()\n                tag_case = (trace.info.tags or {}).get(\'case_id\')\n                if tag_case in expected_cases:\n                    candidates.add(tag_case)\n                for span in trace.data.spans:\n                    inputs = span.inputs\n                    if isinstance(inputs, dict) and inputs.get(\'case_id\') in expected_cases:\n                        candidates.add(inputs[\'case_id\'])\n                if len(candidates) != 1:\n                    raise ValueError(\'Cannot uniquely associate an evaluation trace with a dataset case\')\n                case_id = candidates.pop()\n                metadata = getattr(trace.info, \'trace_metadata\', None)\n                if metadata is None:\n                    metadata = getattr(trace.info, \'request_metadata\', {})\n                found[trace.info.trace_id] = {\n                    \'trace_id\': trace.info.trace_id, \'case_id\': case_id,\n                    \'variant\': evaluation[\'variant\'], \'evaluation\': evaluation[\'id\'],\n                    \'session_id\': (metadata or {}).get(\'mlflow.trace.session\'),\n                }\n                seen_cases.add(case_id)\n            # Retain driver-side probes as ordinary traces, not evaluated predictions.\n            for trace_id, entry in links.items():\n                if entry.get(\'evaluation\') == evaluation[\'id\'] and trace_id not in found:\n                    entry[\'evaluation_probe_for\'] = entry.pop(\'evaluation\')\n                    entry[\'evaluation\'] = None\n            links.update(found)\n            missing = sorted(expected_cases - seen_cases)\n            reports.append({\'evaluation_id\': evaluation[\'id\'], \'run_id\': evaluation[\'run_id\'],\n                            \'persisted_trace_count\': len(found), \'covered_cases\': len(seen_cases),\n                            \'expected_cases\': len(expected_cases), \'missing_cases\': missing})\n            if missing:\n                errors.append({\'evaluation_id\': evaluation[\'id\'], \'error_type\': \'MissingEvaluationCases\',\n                               \'missing_cases\': missing})\n        except Exception as exc:\n            errors.append({\'evaluation_id\': evaluation[\'id\'], \'error_type\': type(exc).__name__})\n    return links, reports, errors\n\n\ndef discover_experiment_traces(mlflow, manifest, links):\n    """Include later manual/monitoring traces within this explicitly selected experiment."""\n    found = mlflow.search_traces(locations=[manifest[\'experiment_id\']],\n                                return_type=\'list\', max_results=10000)\n    if len(found) >= 10000:\n        raise ValueError(\'Experiment trace discovery reached 10000; pagination required before export.\')\n    for trace in found:\n        tags = trace.info.tags or {}\n        metadata = trace.info.trace_metadata or {}\n        links.setdefault(trace.info.trace_id, {\n            \'trace_id\': trace.info.trace_id, \'case_id\': tags.get(\'case_id\'),\n            \'variant\': tags.get(\'variant\'), \'evaluation\': None,\n            \'session_id\': metadata.get(\'mlflow.trace.session\'), \'discovered\': True})\n    return len(found)\n\n\ndef export(manifest_path, output_path, tracking_uri=None, discover_traces=False):\n    import mlflow\n    manifest_path = Path(manifest_path)\n    manifest = json.loads(manifest_path.read_text())\n    if manifest[\'schema_version\'] != SCHEMA_VERSION:\n        raise ValueError(\'Unsupported manifest schema\')\n    effective_tracking_uri = tracking_uri or manifest[\'tracking_uri\']\n    mlflow.set_tracking_uri(effective_tracking_uri)\n    if is_databricks_uri(effective_tracking_uri):\n        experiment = mlflow.get_experiment(manifest[\'experiment_id\'])\n        warehouse = os.environ.get(\'MLFLOW_TRACING_SQL_WAREHOUSE_ID\') or (experiment.tags or {}).get(\'mlflow.monitoring.sqlWarehouseId\')\n        if warehouse:\n            os.environ[\'MLFLOW_TRACING_SQL_WAREHOUSE_ID\'] = warehouse\n        elif getattr(experiment, \'trace_location\', None) is not None:\n            raise ValueError(\'UC export requires MLFLOW_TRACING_SQL_WAREHOUSE_ID or the experiment monitoring warehouse tag.\')\n    bundle = {\'schema_version\': SCHEMA_VERSION,\n              \'source\': {k: manifest[k] for k in (\'seed_id\', \'experiment_id\', \'experiment_name\', \'tracking_uri\', \'mlflow_version\')},\n              \'inventory\': deepcopy(manifest[\'resources\']), \'dataset\': deepcopy(manifest[\'dataset\']),\n              \'prompts\': deepcopy(manifest[\'prompts\']), \'scorers\': deepcopy(manifest[\'scorers\']),\n              \'evaluations\': deepcopy(manifest[\'evaluations\']), \'traces\': [], \'export_errors\': []}\n    bundle[\'source_extensions\'] = {k: deepcopy(manifest[k]) for k in\n        (\'agent_versions\', \'review_session\', \'previous_evaluations\', \'extension\', \'tab_report\') if k in manifest}\n    def artifact(name):\n        local = mlflow.artifacts.download_artifacts(run_id=manifest[\'artifact_run_id\'], artifact_path=\'seed/\' + name)\n        return json.loads(Path(local).read_text())\n    # Read persisted artifacts rather than trusting in-memory seed fixtures.\n    dataset_artifact = artifact(\'dataset.json\')[\'records\']\n    bundle[\'dataset\'][\'records\'] = dataset_artifact\n    bundle[\'scorers\'] = artifact(\'scorers.json\')[\'scorers\']\n    native = bundle[\'dataset\'].get(\'native\')\n    if native:\n        ds = mlflow.genai.datasets.get_dataset(name=native[\'name\']) if is_databricks_uri(effective_tracking_uri) else mlflow.genai.datasets.get_dataset(dataset_id=native[\'id\'])\n        native_rows = plain(ds.to_df().to_dict(orient=\'records\'))\n        by_inputs = {digest(r[\'inputs\']): r for r in dataset_artifact}\n        merged = []\n        for r in native_rows:\n            key = digest(r[\'inputs\'])\n            if key not in by_inputs:\n                raise ValueError(\'Native dataset has an unmanifested row; expand source manifest explicitly.\')\n            base = deepcopy(by_inputs[key])\n            base[\'inputs\'], base[\'expectations\'] = r[\'inputs\'], r[\'expectations\']\n            base[\'native_record\'] = r\n            merged.append(base)\n        if len(merged) != len(dataset_artifact):\n            raise ValueError(\'Native dataset count differs from the seed artifact; review before export.\')\n        bundle[\'dataset\'][\'records\'] = merged\n    persisted_prompts = {p[\'version\']: p for p in artifact(\'prompts.json\')[\'versions\']}\n    for p in bundle[\'prompts\']:\n        if p.get(\'native\'):\n            if is_databricks_uri(effective_tracking_uri):\n                mlflow.set_registry_uri(databricks_registry_uri(effective_tracking_uri))\n            native = p[\'native\']\n            pv = mlflow.genai.load_prompt(f"prompts:/{native[\'name\']}/{native[\'version\']}")\n            p[\'template\'] = plain(pv.template)\n        else:\n            p[\'template\'] = persisted_prompts[p[\'version\']][\'template\']\n    for e in bundle[\'evaluations\']:\n        if e.get(\'run_id\'):\n            run = mlflow.MlflowClient().get_run(e[\'run_id\'])\n            e[\'metrics\'] = plain(run.data.metrics)\n            e[\'tags\'] = plain(run.data.tags)\n        if e.get(\'results_artifact\'):\n            local = mlflow.artifacts.download_artifacts(\n                run_id=manifest[\'artifact_run_id\'], artifact_path=e[\'results_artifact\'])\n            e[\'rows\'] = json.loads(Path(local).read_text())\n        elif e.get(\'results_file\'):\n            local = manifest_path.parent / e[\'results_file\']\n            if not local.is_file():\n                raise FileNotFoundError(\'Older seed manifest: copy its evaluation-*.json files alongside the manifest, or regenerate with the current seed.\')\n            e[\'rows\'] = json.loads(local.read_text())\n    links, reports, errors = reconcile_evaluation_links(mlflow, manifest)\n    if discover_traces:\n        bundle[\'discovered_trace_count\'] = discover_experiment_traces(mlflow, manifest, links)\n        from mlflow.genai.scorers import list_scorers\n        bundle[\'registered_scorers\'] = [\n            {\'name\': s.name, \'sample_rate\': s.sample_rate, \'filter_string\': s.filter_string}\n            for s in list_scorers(experiment_id=manifest[\'experiment_id\'])]\n    bundle[\'evaluation_trace_reconciliation\'] = reports\n    bundle[\'export_errors\'].extend(errors)\n    for trace_id, link in links.items():\n        try:\n            trace = None\n            for attempt in range(6):\n                trace = mlflow.get_trace(trace_id)\n                if trace is not None:\n                    break\n                time.sleep(2)\n            if trace is None:\n                raise LookupError(\'Trace not available\')\n            bundle[\'traces\'].append(normalize_trace(trace, link))\n        except Exception as exc:\n            bundle[\'export_errors\'].append({\'trace_id\': trace_id, \'error_type\': type(exc).__name__})\n    save_json(output_path, bundle)\n    print(f\'Exported {len(bundle["traces"])} traces; {len(bundle["export_errors"])} errors. Bundle: {output_path}\')\n    if bundle[\'export_errors\']:\n        raise RuntimeError(\'Export incomplete. Migration refuses this bundle; retry export after resolving errors.\')\n    return bundle\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--manifest\', required=True)\n    parser.add_argument(\'--output\', required=True)\n    parser.add_argument(\'--tracking-uri\', help=\'Override source authentication routing, e.g. databricks://PROFILE\')\n    parser.add_argument(\'--discover-traces\', action=\'store_true\', help=\'Include all traces in the selected experiment (bounded at 10000).\')\n    args = parser.parse_args()\n    export(args.manifest, args.output, args.tracking_uri, args.discover_traces)\n', 'migration_core.py': '"""Pure transformation layer; offline validation and plans require only Python."""\nfrom datetime import datetime, timezone\nimport re\nfrom uuid import UUID, uuid5\nfrom stack_common import SCHEMA_VERSION, digest\n\nNAMESPACE = UUID(\'d789cfb3-1a4a-4daf-a9e8-b2ad810188a8\')\nTYPE_MAP = {\'AGENT\': \'chain\', \'CHAIN\': \'chain\', \'TOOL\': \'tool\', \'RETRIEVER\': \'retriever\',\n            \'LLM\': \'llm\', \'CHAT_MODEL\': \'llm\', \'EMBEDDING\': \'embedding\',\n            \'PARSER\': \'parser\', \'RERANKER\': \'chain\', \'UNKNOWN\': \'chain\'}\n\n\ndef stable_id(source, kind, key):\n    return str(uuid5(NAMESPACE, f\'{source["seed_id"]}/{kind}/{key}\'))\n\n\ndef timestamp(ns):\n    if not isinstance(ns, int) or ns <= 0:\n        raise ValueError(\'Missing or invalid original nanosecond timestamp\')\n    seconds, nanos = divmod(ns, 1_000_000_000)\n    return datetime.fromtimestamp(seconds, timezone.utc).replace(microsecond=nanos // 1000)\n\n\ndef object_io(value):\n    return value if isinstance(value, dict) else {\'value\': value}\n\n\ndef convert_prompt(template):\n    if not isinstance(template, str) or \'{%\' in template:\n        raise ValueError(\'Only simple text prompts with {{identifier}} variables are supported\')\n    pattern = r\'\\{\\{\\s*([A-Za-z_][A-Za-z_0-9]*)\\s*\\}\\}\'\n    remainder = re.sub(pattern, \'\', template)\n    if \'{\' in remainder or \'}\' in remainder:\n        raise ValueError(\'Complex or literal prompt braces require manual conversion\')\n    return re.sub(pattern, r\'{\\1}\', template)\n\n\ndef ordered_spans(trace):\n    spans = trace[\'spans\']\n    by_id = {s[\'id\']: s for s in spans}\n    if len(by_id) != len(spans):\n        raise ValueError(\'Duplicate span ID\')\n    roots = [s for s in spans if not s[\'parent_id\']]\n    if len(roots) != 1:\n        raise ValueError(\'A complete trace must have exactly one root\')\n    visiting, seen, ordered = set(), set(), []\n    def visit(span):\n        if span[\'id\'] in seen:\n            return\n        if span[\'id\'] in visiting:\n            raise ValueError(\'Span hierarchy contains a cycle\')\n        visiting.add(span[\'id\'])\n        if span[\'parent_id\']:\n            if span[\'parent_id\'] not in by_id:\n                raise ValueError(\'Orphan span; refusing to flatten hierarchy\')\n            visit(by_id[span[\'parent_id\']])\n        visiting.remove(span[\'id\'])\n        seen.add(span[\'id\'])\n        ordered.append(span)\n    for span in sorted(spans, key=lambda s: (s[\'start_ns\'], s[\'id\'])):\n        visit(span)\n    return ordered\n\n\ndef plan_bundle(bundle, prefix=\'mlflow-demo\'):\n    if bundle.get(\'schema_version\') != SCHEMA_VERSION:\n        raise ValueError(\'Unsupported bundle schema\')\n    if bundle.get(\'export_errors\'):\n        raise ValueError(\'Incomplete source export\')\n    if not re.fullmatch(r\'[a-z0-9][a-z0-9-]{0,39}\', prefix):\n        raise ValueError(\'Prefix must be lowercase letters/digits/hyphens, max 40 characters\')\n    source = bundle[\'source\']\n    id_source = {**source, \'seed_id\': source[\'seed_id\'] + \'/\' + prefix}\n    namespace = prefix + \'-\' + source[\'seed_id\'][:12]\n    dataset = bundle[\'dataset\']\n    examples = []\n    row_ids = set()\n    for row in dataset[\'records\']:\n        if row[\'id\'] in row_ids:\n            raise ValueError(\'Duplicate dataset row ID\')\n        if not isinstance(row[\'inputs\'], dict) or not isinstance(row[\'expectations\'], dict):\n            raise ValueError(\'Dataset inputs and expectations must be dictionaries\')\n        row_ids.add(row[\'id\'])\n        examples.append({\'id\': stable_id(id_source, \'example\', row[\'id\']), \'source_id\': row[\'id\'],\n                         \'inputs\': row[\'inputs\'], \'outputs\': row[\'expectations\'],\n                         \'metadata\': {**row.get(\'metadata\', {}), \'mlflow_source_row\': row[\'id\']}})\n    evaluations = {e[\'id\']: e for e in bundle[\'evaluations\']}\n    projects = {namespace + \'-traces\': {\'kind\': \'tracing\', \'metadata\': {\'mlflow_source\': source}}}\n    if source.get(\'discovery\') and not bundle[\'traces\']:\n        projects = {}\n    for e in evaluations.values():\n        if e[\'status\'] == \'created\':\n            projects[namespace + \'-eval-\' + e[\'variant\']] = {\n                \'kind\': \'evaluation\', \'metadata\': {\'mlflow_source\': source,\n                \'source_evaluation\': {k: v for k, v in e.items() if k != \'rows\'}}}\n    runs, feedback, skipped = [], [], []\n    seen_trace_ids, seen_assessments = set(), set()\n    for trace in bundle[\'traces\']:\n        if trace[\'id\'] in seen_trace_ids:\n            raise ValueError(\'Duplicate source trace ID\')\n        seen_trace_ids.add(trace[\'id\'])\n        spans = ordered_spans(trace)\n        root = next(s for s in spans if not s[\'parent_id\'])\n        root_id = stable_id(id_source, \'run\', trace[\'id\'] + \'/\' + root[\'id\'])\n        ids = {s[\'id\']: stable_id(id_source, \'run\', trace[\'id\'] + \'/\' + s[\'id\']) for s in spans}\n        orders = {}\n        evaluation_id = trace[\'link\'].get(\'evaluation\')\n        e = evaluations.get(evaluation_id)\n        if evaluation_id and e is None:\n            raise ValueError(\'Trace references an unknown evaluation\')\n        project = namespace + (\'-eval-\' + e[\'variant\'] if e and e[\'status\'] == \'created\' else \'-traces\')\n        reference_id = None\n        if e and e[\'status\'] == \'created\':\n            case_id = trace[\'link\'][\'case_id\']\n            if case_id not in row_ids:\n                raise ValueError(\'Evaluation trace references a missing example\')\n            reference_id = stable_id(id_source, \'example\', case_id)\n        for s in spans:\n            start, end = timestamp(s[\'start_ns\']), timestamp(s[\'end_ns\'])\n            if end < start:\n                raise ValueError(\'Span ends before it starts\')\n            segment = start.strftime(\'%Y%m%dT%H%M%S%fZ\') + ids[s[\'id\']]\n            orders[s[\'id\']] = ((orders[s[\'parent_id\']] + \'.\') if s[\'parent_id\'] else \'\') + segment\n            metadata = {\'mlflow_trace_id\': trace[\'id\'], \'mlflow_span_id\': s[\'id\'],\n                        \'mlflow_span_type\': s[\'type\'], \'mlflow_attributes\': s[\'attributes\'],\n                        \'mlflow_trace_metadata\': trace[\'metadata\'], \'mlflow_trace_tags\': trace[\'tags\'],\n                        \'mlflow_link\': trace[\'link\'], \'source_start_ns\': s[\'start_ns\'], \'source_end_ns\': s[\'end_ns\']}\n            if trace.get(\'session_id\'):\n                metadata[\'thread_id\'] = trace[\'session_id\']\n            if s[\'type\'] not in TYPE_MAP:\n                skipped.append({\'kind\': \'span_type\', \'id\': s[\'id\'], \'reason\': f"Unknown type {s[\'type\']} mapped to chain"})\n            runs.append({\'id\': ids[s[\'id\']], \'trace_id\': root_id,\n                         \'parent_run_id\': ids.get(s[\'parent_id\']), \'name\': s[\'name\'],\n                         \'run_type\': TYPE_MAP.get(s[\'type\'], \'chain\'), \'inputs\': object_io(s[\'inputs\']),\n                         \'outputs\': object_io(s[\'outputs\']) if s[\'outputs\'] is not None else None,\n                         \'start_time\': start.isoformat(), \'end_time\': end.isoformat(),\n                         \'dotted_order\': orders[s[\'id\']], \'project_name\': project,\n                         \'error\': (s[\'error\'] or \'Source span status ERROR\') if s[\'status\'] == \'ERROR\' else None,\n                         \'extra\': {\'metadata\': metadata}, \'tags\': ([\'migrated:mlflow\'] if source.get(\'discovery\') else [\'migrated:mlflow\', \'synthetic:demo\']),\n                         \'reference_example_id\': reference_id if s[\'id\'] == root[\'id\'] else None})\n        for a in trace.get(\'assessments\', []):\n            reason = None\n            if a[\'kind\'] != \'feedback\': reason = \'Only feedback assessments are mapped; raw assessment preserved in bundle\'\n            elif a.get(\'valid\') is False or a.get(\'overrides\'): reason = \'Invalidated/override assessment requires manual history mapping\'\n            elif a.get(\'error\'): reason = \'Scorer error is preserved in bundle, not converted into a score\'\n            elif isinstance(a.get(\'value\'), list) or a.get(\'value\') is None: reason = \'Unsupported or missing feedback value\'\n            elif a.get(\'span_id\') and a[\'span_id\'] not in ids: reason = \'Assessment references a missing span\'\n            elif not a.get(\'id\'): reason = \'Assessment has no stable source ID\'\n            if reason:\n                skipped.append({\'kind\': \'assessment\', \'id\': a.get(\'id\'), \'reason\': reason})\n                continue\n            key = trace[\'id\'] + \'/\' + a[\'id\']\n            if key in seen_assessments:\n                raise ValueError(\'Duplicate assessment ID within a trace\')\n            seen_assessments.add(key)\n            value = a[\'value\']\n            feedback.append({\'feedback_id\': stable_id(id_source, \'feedback\', key),\n                             \'run_id\': ids[a[\'span_id\']] if a.get(\'span_id\') else root_id,\n                             \'key\': a[\'name\'], \'score\': value if isinstance(value, (bool, int, float)) else None,\n                             \'value\': value if isinstance(value, (str, dict)) else None,\n                             \'comment\': a.get(\'rationale\'), \'feedback_source_type\': \'api\',\n                             \'source_info\': {\'mlflow_assessment_id\': a[\'id\'], \'mlflow_source\': a[\'source\'],\n                                             \'mlflow_metadata\': a.get(\'metadata\'), \'mlflow_raw\': a.get(\'raw\')}})\n    prompts = []\n    for p in bundle[\'prompts\']:\n        try:\n            converted = convert_prompt(p[\'template\'])\n        except ValueError as exc:\n            skipped.append({\'kind\': \'prompt\', \'id\': p[\'version\'], \'reason\': str(exc)})\n            continue\n        prompt_key = digest([p[\'name\'], p[\'version\']])[:16] if source.get(\'discovery\') else \'support-v\' + p[\'version\']\n        prompts.append({\'identifier\': namespace + \'-\' + prompt_key,\n                        \'template\': converted, \'source\': p})\n    return {\'schema_version\': SCHEMA_VERSION, \'source\': source, \'bundle_digest\': digest(bundle),\n            \'prefix\': prefix, \'dataset_name\': namespace + \'-dataset\', \'examples\': examples,\n            \'create_dataset\': bool(examples) or \'native\' in dataset or not source.get(\'discovery\'),\n            \'projects\': projects, \'runs\': runs, \'feedback\': feedback, \'prompts\': prompts,\n            \'scorers\': bundle[\'scorers\'], \'skipped\': skipped, \'source_inventory\': bundle[\'inventory\'],\n            \'evaluation_trace_reconciliation\': bundle.get(\'evaluation_trace_reconciliation\', []),\n            \'source_evaluations\': [{k: v for k, v in e.items() if k != \'rows\'} for e in bundle[\'evaluations\']],\n            \'limitations\': [\n                (\'Experiment-scoped discovery; historical evaluation relationships are archived, not recreated.\' if source.get(\'discovery\') else \'Inventory-scoped snapshot; linked datasets and prompts are selected by manifest.\'),\n                \'Prompt versions become separate private prompt snapshots; aliases and native version history are not recreated.\',\n                (\'Scorer names and sampling/filter configuration are archived; scorer code is not discovered or recreated.\' if source.get(\'discovery\') else \'Custom scorer source is archived only. The three known demo scorers have hand-written LangSmith adapters.\'),\n                (\'Historical evaluation runs remain source metadata; imported traces do not recreate LangSmith evaluation experiments.\' if source.get(\'discovery\') else \'Evaluation projects link imported trace roots to reference examples; source aggregate metrics remain metadata.\'),\n                \'Nanosecond timestamps are truncated to microseconds; original values remain metadata.\',\n                \'Session IDs map to metadata.thread_id; UI conversation rendering is deployment-dependent.\',\n                \'Token/cost fields remain source attributes; no normalized LangSmith cost accounting is claimed.\',\n                \'Assessment authors/timestamps remain provenance; destination feedback is created by the migration API identity.\',\n                \'Binary attachments, model registry, serving, permissions, schedules, and online scorer configuration are not migrated.\',\n            ]}\n\n\ndef replay_plan(plan, anchor):\n    """Replay each trace near a fixed UTC anchor, retaining intra-trace timing."""\n    from copy import deepcopy\n    from datetime import timedelta\n    anchor = datetime.fromisoformat(anchor)\n    if anchor.tzinfo is None:\n        raise ValueError(\'Replay anchor must include a timezone\')\n    anchor = anchor.astimezone(timezone.utc)\n    result = deepcopy(plan)\n    policy = {\'mode\': \'replay\', \'anchor\': anchor.isoformat()}\n    result[\'timestamp_policy\'] = policy\n    result[\'bundle_digest\'] = digest({\'source_bundle\': plan[\'bundle_digest\'], \'timestamp_policy\': policy})\n    groups = {}\n    for run in result[\'runs\']:\n        groups.setdefault(run[\'trace_id\'], []).append(run)\n    for index, (_, runs) in enumerate(sorted(groups.items())):\n        latest = max(datetime.fromisoformat(r[\'end_time\']) for r in runs)\n        earliest = min(datetime.fromisoformat(r[\'start_time\']) for r in runs)\n        offset = anchor - timedelta(seconds=index) - latest\n        if latest - earliest + timedelta(seconds=index) >= timedelta(hours=23):\n            raise ValueError(\'Replay exceeds the 23-hour ingestion budget; split this export\')\n        orders = {}\n        for run in runs:\n            metadata = run[\'extra\'][\'metadata\']\n            metadata.update({\'source_start_time\': run[\'start_time\'], \'source_end_time\': run[\'end_time\'],\n                             \'migration_timestamp_policy\': policy})\n            for key in (\'start_time\', \'end_time\'):\n                run[key] = (datetime.fromisoformat(run[key]) + offset).isoformat()\n            segment = datetime.fromisoformat(run[\'start_time\']).strftime(\'%Y%m%dT%H%M%S%fZ\') + run[\'id\']\n            parent = run[\'parent_run_id\']\n            orders[run[\'id\']] = (orders[parent] + \'.\' if parent else \'\') + segment\n            run[\'dotted_order\'] = orders[run[\'id\']]\n    result[\'limitations\'].append(\'Replay dates are ingestion dates; original dates remain metadata. Intra-trace durations are preserved; gaps between traces are not.\')\n    return result\n\n\ndef prepare_plans(bundles, prefix, output_dir, replay=False):\n    """Persist a replay anchor so previews, writes and retries use identical dates."""\n    from pathlib import Path\n    import json\n    from stack_common import save_json\n    policy_path = Path(output_dir) / \'timestamp-policy.json\'\n    identity = {\'bundle_digests\': [digest(b) for b in bundles], \'prefix\': prefix, \'replay\': replay}\n    if policy_path.exists():\n        policy = json.loads(policy_path.read_text())\n        if policy[\'identity\'] != identity:\n            raise ValueError(\'Timestamp policy belongs to another bundle, prefix or mode; use a separate output directory\')\n    else:\n        policy = {\'identity\': identity, \'anchor\': datetime.now(timezone.utc).isoformat() if replay else None}\n        save_json(policy_path, policy)\n    plans = [plan_bundle(b, prefix=prefix) for b in bundles]\n    return [replay_plan(p, policy[\'anchor\']) for p in plans] if replay else plans\n', 'migrate_mlflow.py': '"""Plan by default; --apply writes the reviewed plan to LangSmith."""\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport argparse\nimport json\nimport os\nimport time\nfrom stack_common import digest, save_json\nfrom migration_core import plan_bundle\nfrom providers.mlflow.data.datasets import migrate_datasets\nfrom providers.mlflow.data.prompts import migrate_prompts\nfrom providers.mlflow.data.traces import migrate_traces\n\n\nfrom utils.mlflow_destination import LangSmithDestination, assert_run_matches\n\n\ndef apply_plan(plan, destination, state_path, include_prompts=False):\n    state_path = Path(state_path)\n    identity = {\'bundle_digest\': plan[\'bundle_digest\'], \'prefix\': plan[\'prefix\'],\n                \'destination\': destination.identity}\n    if state_path.exists():\n        state = json.loads(state_path.read_text())\n        if state[\'identity\'] != identity:\n            raise ValueError(\'State belongs to another bundle, prefix, or destination\')\n    else:\n        state = {\'identity\': identity, \'completed\': {}, \'status\': \'running\'}\n    state[\'status\'] = \'running\'\n    save_json(state_path, state)\n    def perform(key, operation):\n        if key in state[\'completed\']:\n            return state[\'completed\'][key]\n        result = operation()\n        state[\'completed\'][key] = result\n        save_json(state_path, state)\n        return result\n    dataset_id = migrate_datasets(plan, destination, perform)\n    migrate_traces(plan, destination, perform, dataset_id)\n    if include_prompts:\n        migrate_prompts(plan, destination, perform)\n    for name, item in plan[\'projects\'].items():\n        if item[\'kind\'] == \'evaluation\':\n            project_id = state[\'completed\'][\'project:\' + name]\n            perform(\'finalize:\' + name, lambda p=project_id: destination.finalize(p))\n    state[\'status\'] = \'applied_requires_validation\'\n    state[\'prompts_applied\'] = bool(include_prompts or state.get(\'prompts_applied\'))\n    save_json(state_path, state)\n    return state\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--bundle\', required=True)\n    parser.add_argument(\'--prefix\', default=\'mlflow-demo\')\n    parser.add_argument(\'--report\', default=\'migration-plan.json\')\n    parser.add_argument(\'--state\', default=\'migration-state.json\')\n    parser.add_argument(\'--apply\', action=\'store_true\')\n    parser.add_argument(\'--include-prompts\', action=\'store_true\', help=\'Requires langchain-core; creates version snapshots\')\n    args = parser.parse_args()\n    bundle = json.loads(Path(args.bundle).read_text())\n    plan = plan_bundle(bundle, args.prefix)\n    save_json(args.report, plan)\n    print({k: len(plan[k]) for k in (\'examples\', \'projects\', \'runs\', \'feedback\', \'prompts\', \'skipped\')})\n    if args.apply:\n        apply_plan(plan, LangSmithDestination(), args.state, args.include_prompts)\n        print(\'Applied supported resources. Run validate_migration.py before claiming completion.\')\n    else:\n        print(\'Dry run only. Review source_inventory, skipped, and limitations in \' + args.report)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'validate_migration.py': '"""Validate a bundle offline, or read back migrated resources with --live."""\nfrom pathlib import Path\nimport argparse\nimport json\nfrom migration_core import plan_bundle\nfrom stack_common import save_json, digest\nfrom migrate_mlflow import LangSmithDestination, assert_run_matches\n\n\ndef validate_live(plan, destination, state):\n    if state[\'identity\'] != {\'bundle_digest\': plan[\'bundle_digest\'], \'prefix\': plan[\'prefix\'], \'destination\': destination.identity}:\n        raise ValueError(\'Checkpoint identity does not match this validation target\')\n    client = destination.client\n    checked = {\'examples\': 0, \'projects\': 0, \'runs\': 0, \'feedback\': 0, \'prompts\': 0}\n    errors = []\n    def check(kind, key, operation):\n        try:\n            operation()\n            checked[kind] += 1\n        except Exception as exc:\n            errors.append({\'kind\': kind, \'id\': key, \'error_type\': type(exc).__name__})\n    ds = None\n    if plan.get(\'create_dataset\', True):\n        ds = client.read_dataset(dataset_name=plan[\'dataset_name\'])\n        if (ds.metadata or {}).get(\'migration_bundle_digest\') != plan[\'bundle_digest\']:\n            errors.append({\'kind\': \'dataset\', \'id\': str(ds.id), \'error_type\': \'DatasetProvenanceMismatch\'})\n    def verify_example(item):\n        obj = client.read_example(example_id=item[\'id\'])\n        assert str(obj.dataset_id) == str(ds.id)\n        assert obj.inputs == item[\'inputs\'] and obj.outputs == item[\'outputs\']\n        assert all((obj.metadata or {}).get(k) == v for k, v in item[\'metadata\'].items())\n    if ds is not None:\n        actual_ids = {str(e.id) for e in client.list_examples(dataset_id=ds.id)}\n        if actual_ids != {e[\'id\'] for e in plan[\'examples\']}:\n            errors.append({\'kind\': \'dataset\', \'id\': str(ds.id), \'error_type\': \'ExampleSetMismatch\'})\n    projects = {}\n    actual_runs = {}\n    for name, item in plan[\'projects\'].items():\n        def verify_project(n=name, i=item):\n            obj = client.read_project(project_name=n)\n            projects[n] = obj\n            assert (obj.extra or {}).get(\'metadata\', {}).get(\'migration_bundle_digest\') == plan[\'bundle_digest\']\n            expected_reference = str(ds.id) if i[\'kind\'] == \'evaluation\' else None\n            assert (str(obj.reference_dataset_id) if obj.reference_dataset_id else None) == expected_reference\n            if i[\'kind\'] == \'evaluation\':\n                assert obj.end_time is not None\n            rows = list(client.list_runs(project_name=n))\n            actual_runs.update({str(r.id): r for r in rows})\n            ids = {str(r.id) for r in rows}\n            assert ids == {r[\'id\'] for r in plan[\'runs\'] if r[\'project_name\'] == n}\n        check(\'projects\', name, verify_project)\n    for item in plan[\'examples\']:\n        check(\'examples\', item[\'id\'], lambda i=item: verify_example(i))\n    for item in plan[\'runs\']:\n        def verify_run(i=item):\n            obj = actual_runs[i[\'id\']]\n            assert_run_matches(obj, i)\n            assert str(obj.session_id) == str(projects[i[\'project_name\']].id)\n        check(\'runs\', item[\'id\'], verify_run)\n    actual_feedback = {}\n    feedback_runs = sorted({i[\'run_id\'] for i in plan[\'feedback\']})\n    for start in range(0, len(feedback_runs), 50):\n        for obj in client.list_feedback(run_ids=feedback_runs[start:start + 50]):\n            actual_feedback[str(obj.id)] = obj\n    for item in plan[\'feedback\']:\n        def verify_feedback(i=item):\n            obj = actual_feedback[i[\'feedback_id\']]\n            assert str(obj.run_id) == i[\'run_id\']\n            assert all(getattr(obj, k) == i[k] for k in (\'key\', \'score\', \'value\', \'comment\'))\n        check(\'feedback\', item[\'feedback_id\'], verify_feedback)\n    if state.get(\'prompts_applied\'):\n        for item in plan[\'prompts\']:\n            def verify_prompt(i=item):\n                obj = client.pull_prompt(i[\'identifier\'])\n                assert obj.template == i[\'template\']\n            check(\'prompts\', item[\'identifier\'], verify_prompt)\n    return {\'mode\': \'live_readback\', \'passed\': not errors, \'checked\': checked,\n            \'errors\': errors, \'skipped\': plan[\'skipped\'], \'limitations\': plan[\'limitations\']}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--bundle\', required=True)\n    parser.add_argument(\'--prefix\', default=\'mlflow-demo\')\n    parser.add_argument(\'--live\', action=\'store_true\')\n    parser.add_argument(\'--state\', default=\'migration-state.json\')\n    parser.add_argument(\'--report\', default=\'validation-report.json\')\n    args = parser.parse_args()\n    bundle = json.loads(Path(args.bundle).read_text())\n    plan = plan_bundle(bundle, args.prefix)\n    if args.live:\n        state = json.loads(Path(args.state).read_text())\n        result = validate_live(plan, LangSmithDestination(), state)\n    else:\n        result = {\'mode\': \'offline_structure_only\', \'passed\': True,\n                  \'bundle_digest\': digest(bundle), \'trace_count\': len(bundle[\'traces\']),\n                  \'span_count\': len(plan[\'runs\']), \'destination_verified\': False,\n                  \'skipped\': plan[\'skipped\'], \'limitations\': plan[\'limitations\']}\n    save_json(args.report, result)\n    print(f\'{result["mode"]}: {"PASS" if result["passed"] else "FAIL"}; {args.report}\')\n    if not result[\'passed\']:\n        raise SystemExit(1)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'discover_mlflow.py': '"""Read-only, experiment-scoped discovery, independent of the setup inventory."""\nfrom copy import deepcopy\nimport os\n\nfrom stack_common import SCHEMA_VERSION, digest, plain, save_json, is_databricks_uri, databricks_registry_uri\n\n\nfrom utils.mlflow import bounded, connect\nfrom providers.mlflow.data.datasets import export_datasets\nfrom providers.mlflow.data.prompts import export_prompts\nfrom providers.mlflow.data.traces import export_traces\n\n\ndef list_experiments(tracking_uri=\'databricks\'):\n    _, client = connect(tracking_uri)\n    rows, token = [], None\n    while True:\n        page = client.search_experiments(max_results=100, page_token=token)\n        rows.extend({\'id\': e.experiment_id, \'name\': e.name} for e in page)\n        token = getattr(page, \'token\', None)\n        if not token:\n            return rows\n\n\ndef export_experiment(experiment_id, output_path, tracking_uri=\'databricks\',\n                      dataset_names=(), prompt_names=(), limit=10000, source_key=None):\n    """Export a portable collection: one bundle per dataset and one for traces/prompts.\n\n    Explicit dataset/prompt names augment discovery; they never suppress discovery errors.\n    Unsupported APIs are coverage gaps, not silently reported as empty collections.\n    """\n    if limit < 2:\n        raise ValueError(\'limit must be at least 2\')\n    mlflow, client = connect(tracking_uri)\n    experiment = client.get_experiment(str(experiment_id))\n    # Include source host in identity; profile aliases alone are not stable identities.\n    if source_key is None:\n        if is_databricks_uri(tracking_uri):\n            from databricks.sdk import WorkspaceClient\n            from mlflow.utils.uri import get_db_info_from_uri\n            profile, _ = get_db_info_from_uri(tracking_uri)\n            ws = WorkspaceClient(profile=profile) if profile else WorkspaceClient()\n            source_key = ws.config.host\n        else:\n            source_key = tracking_uri\n    warehouse = os.environ.get(\'MLFLOW_TRACING_SQL_WAREHOUSE_ID\') or (experiment.tags or {}).get(\'mlflow.monitoring.sqlWarehouseId\')\n    if warehouse:\n        os.environ[\'MLFLOW_TRACING_SQL_WAREHOUSE_ID\'] = warehouse\n    source = {\'seed_id\': digest([source_key, str(experiment_id)])[:24],\n              \'experiment_id\': str(experiment_id), \'experiment_name\': experiment.name,\n              \'tracking_uri\': tracking_uri, \'mlflow_version\': mlflow.__version__,\n              \'discovery\': True}\n    def empty_bundle(identity, name):\n        return {\'schema_version\': SCHEMA_VERSION, \'source\': {**source, \'seed_id\': identity},\n                \'inventory\': {}, \'dataset\': {\'name\': name, \'records\': []},\n                \'prompts\': [], \'scorers\': [], \'evaluations\': [], \'traces\': [], \'export_errors\': []}\n    primary = empty_bundle(source[\'seed_id\'], \'trace-export\')\n    result = {\'format\': \'mlflow-discovery-v1\', \'source\': source, \'coverage\': {},\n              \'bundles\': [primary], \'export_errors\': [],\n              \'limitations\': [\n                  \'Evaluation/training run metadata is archived; historical experiment/example relationships are not reconstructed.\',\n                  \'Each dataset remains separate. Dataset historical versions and arbitrary run artifacts are not exported.\',\n                  \'Agent models, review workflow configuration, online schedules, serving and permissions are not recreated.\',\n                  \'Prompt versions become separate private snapshots; aliases remain source metadata.\',\n                  \'UC prompt discovery follows run/trace links and explicit names; unlinked registry prompts are outside this experiment export.\',\n                  \'Discovery is bounded and refuses truncated results; it does not silently sample.\']}\n    def read(kind, fn):\n        try:\n            count = fn()\n            result[\'coverage\'][kind] = {\'status\': \'exported\', \'count\': count}\n        except Exception as exc:\n            error = {\'resource\': kind, \'error_type\': type(exc).__name__}\n            result[\'export_errors\'].append(error)\n            result[\'coverage\'][kind] = {\'status\': \'failed\', **error}\n\n    def traces():\n        primary[\'traces\'] = export_traces(mlflow, experiment_id, limit)\n        return len(primary[\'traces\'])\n    read(\'traces_and_assessments\', traces)\n\n    def datasets():\n        rows = export_datasets(mlflow, experiment_id, tracking_uri, dataset_names, limit)\n        for dataset in rows:\n            identity = digest([source[\'seed_id\'], \'dataset\', dataset[\'name\']])[:24]\n            bundle = empty_bundle(identity, dataset[\'name\'])\n            bundle[\'dataset\'] = dataset\n            result[\'bundles\'].append(bundle)\n        return len(rows)\n    read(\'datasets\', datasets)\n\n    def prompts():\n        primary[\'prompts\'] = export_prompts(mlflow, client, experiment_id, prompt_names, limit)\n        return len(primary[\'prompts\'])\n    read(\'prompt_versions\', prompts)\n\n    def runs():\n        rows = bounded(client.search_runs([str(experiment_id)], max_results=limit), limit, \'runs\')\n        result[\'source_runs\'] = [plain(run) for run in rows]\n        return len(rows)\n    read(\'run_metadata\', runs)\n\n    def scorers():\n        from mlflow.genai.scorers import list_scorers\n        rows = list_scorers(experiment_id=str(experiment_id))\n        primary[\'scorers\'] = [{\'name\': s.name, \'sample_rate\': s.sample_rate,\n                               \'filter_string\': s.filter_string} for s in rows]\n        return len(rows)\n    read(\'scorer_configuration\', scorers)\n    primary[\'inventory\'] = deepcopy(result[\'coverage\'])\n    for bundle in result[\'bundles\']:\n        bundle[\'export_errors\'] = deepcopy(result[\'export_errors\'])\n    save_json(output_path, result)\n    save_json(str(output_path) + \'.coverage.json\', {k: result[k] for k in (\'coverage\', \'export_errors\', \'limitations\')})\n    if result[\'export_errors\']:\n        raise RuntimeError(\'Discovery incomplete. Inspect the saved coverage report; destination import is blocked.\')\n    return result\n', 'providers/__init__.py': '', 'providers/mlflow/__init__.py': '', 'providers/mlflow/data/__init__.py': '', 'providers/mlflow/data/datasets.py': '"""Discover MLflow datasets and migrate their reference examples."""\nfrom stack_common import digest, plain, is_databricks_uri\nfrom utils.mlflow import bounded\n\n\ndef export_datasets(mlflow, experiment_id, tracking_uri, names=(), limit=10000):\n    found = bounded(mlflow.genai.datasets.search_datasets(\n        experiment_ids=[str(experiment_id)], max_results=limit,\n        **({} if is_databricks_uri(tracking_uri) else {\'filter_string\': \'created_time >= 0\'})),\n        limit, \'datasets\')\n    by_name = {ds.name: ds for ds in found}\n    for name in names:\n        if name not in by_name:\n            by_name[name] = mlflow.genai.datasets.get_dataset(name=name)\n    datasets = []\n    for name, ds in sorted(by_name.items()):\n        # Search returns tracking-store entities; UC rows must be loaded by name.\n        if is_databricks_uri(tracking_uri):\n            ds = mlflow.genai.datasets.get_dataset(name=name)\n        dataset = {\'name\': name, \'native\': {\'name\': name, \'id\': ds.dataset_id}, \'records\': []}\n        for idx, row in enumerate(plain(ds.to_df().to_dict(orient=\'records\'))):\n            inputs, refs = row.get(\'inputs\'), row.get(\'expectations\')\n            if not isinstance(inputs, dict) or not isinstance(refs, dict):\n                raise ValueError(\'Dataset records require dictionary inputs and expectations; predictions are not references.\')\n            dataset[\'records\'].append({\n                \'id\': str(row.get(\'dataset_record_id\') or row.get(\'id\') or digest([idx, row])),\n                \'inputs\': inputs, \'expectations\': refs,\n                \'metadata\': {\'source_dataset\': name}, \'native_record\': row})\n        datasets.append(dataset)\n    return datasets\n\n\ndef migrate_datasets(plan, destination, perform):\n    dataset_id = (perform(\'dataset\', lambda: destination.dataset(plan))\n                  if plan.get(\'create_dataset\', True) else None)\n    for item in plan[\'examples\']:\n        perform(\'example:\' + item[\'id\'], lambda i=item: destination.example(i, dataset_id))\n    return dataset_id\n', 'providers/mlflow/data/prompts.py': '"""Discover prompt versions and migrate them as private snapshots."""\nfrom stack_common import plain\nfrom utils.mlflow import bounded\n\n\ndef export_prompts(mlflow, client, experiment_id, names=(), limit=10000):\n    import json\n    if getattr(mlflow, \'get_registry_uri\', lambda: \'\')().startswith(\'databricks-uc\'):\n        # Managed UC rejects experiment filters. Follow recorded run/trace links\n        # instead of exporting unrelated prompts from the surrounding schema.\n        found_names = set(names)\n        runs = bounded(client.search_runs([str(experiment_id)], max_results=limit), limit, \'prompt-linked runs\')\n        for run in runs:\n            for item in json.loads(run.data.tags.get(\'mlflow.linkedPrompts\', \'[]\')):\n                found_names.add(item[\'name\'])\n        for trace in bounded(mlflow.search_traces(locations=[str(experiment_id)],\n                return_type=\'list\', max_results=limit), limit, \'prompt-linked traces\'):\n            for item in json.loads((trace.info.tags or {}).get(\'mlflow.linkedPrompts\', \'[]\')):\n                found_names.add(item[\'name\'])\n    else:\n        found = bounded(mlflow.genai.search_prompts(\n            filter_string=f"experiment_id = \'{experiment_id}\'", max_results=limit), limit, \'prompts\')\n        found_names = {p.name for p in found} | set(names)\n    result = []\n    for name in sorted(found_names):\n        escaped = name.replace("\'", "\'\'")\n        if hasattr(client, \'search_prompt_versions\'):\n            versions, token = [], None\n            while True:\n                page = client.search_prompt_versions(name, max_results=min(100, limit), page_token=token)\n                versions.extend(page)\n                bounded(versions, limit, \'prompt versions\')\n                token = getattr(page, \'token\', None)\n                if not token:\n                    break\n        else:\n            versions = bounded(client.search_model_versions(\n                filter_string=f"name = \'{escaped}\'", max_results=limit), limit, \'prompt versions\')\n        if not versions:\n            raise ValueError(f\'No readable versions for linked prompt {name}\')\n        for version in versions:\n            pv = client.load_prompt(f\'prompts:/{name}/{version.version}\')\n            result.append({\'name\': name, \'version\': str(version.version),\n                \'template\': plain(pv.template), \'aliases\': list(getattr(version, \'aliases\', []) or [])})\n    return result\n\n\ndef migrate_prompts(plan, destination, perform):\n    for item in plan[\'prompts\']:\n        perform(\'prompt:\' + item[\'identifier\'], lambda i=item: destination.prompt(i))\n', 'providers/mlflow/data/traces.py': '"""Discover traces and migrate projects, span hierarchies, and feedback."""\nfrom utils.mlflow import bounded\n\n\ndef export_traces(mlflow, experiment_id, limit=10000):\n    from export_mlflow import normalize_trace\n    rows = bounded(mlflow.search_traces(locations=[str(experiment_id)],\n        return_type=\'list\', max_results=limit), limit, \'traces\')\n    return [normalize_trace(t, {\'evaluation\': None, \'discovered\': True}) for t in rows]\n\n\ndef migrate_traces(plan, destination, perform, dataset_id):\n    for name, item in plan[\'projects\'].items():\n        perform(\'project:\' + name, lambda n=name, i=item: destination.project(n, i, dataset_id, plan))\n    if hasattr(destination, \'runs\'):\n        destination.runs(plan[\'runs\'], perform)\n    else:\n        for item in plan[\'runs\']:\n            perform(\'run:\' + item[\'id\'], lambda i=item: destination.run(i))\n    for item in plan[\'feedback\']:\n        perform(\'feedback:\' + item[\'feedback_id\'], lambda i=item: destination.feedback(i))\n', 'utils/__init__.py': '', 'utils/mlflow.py': '"""MLflow connection and bounded discovery helpers shared by provider resources."""\nfrom stack_common import is_databricks_uri, databricks_registry_uri\n\ndef bounded(items, limit, resource):\n    rows = list(items)\n    if len(rows) >= limit or getattr(items, \'token\', None):\n        raise ValueError(f\'{resource} reached discovery bound; narrow scope before export.\')\n    return rows\n\n\ndef connect(tracking_uri):\n    import mlflow\n    mlflow.set_tracking_uri(tracking_uri)\n    if is_databricks_uri(tracking_uri):\n        mlflow.set_registry_uri(databricks_registry_uri(tracking_uri))\n    return mlflow, mlflow.MlflowClient()\n\n\n', 'utils/mlflow_destination.py': '"""LangSmith destination access and read-back comparison helpers."""\nfrom datetime import datetime, timezone\nimport os\nimport time\nfrom stack_common import digest\n\n\ndef iso(value):\n    if value is None:\n        return None\n    return value.isoformat() if hasattr(value, \'isoformat\') else datetime.fromisoformat(value).isoformat()\n\n\ndef assert_run_matches(actual, expected):\n    for key in (\'name\', \'run_type\', \'inputs\', \'outputs\', \'error\', \'dotted_order\'):\n        if getattr(actual, key, None) != expected[key]:\n            raise ValueError(f\'Destination run mismatch: {expected["id"]} field {key}\')\n    for key in (\'parent_run_id\', \'trace_id\', \'reference_example_id\'):\n        value = getattr(actual, key, None)\n        if (str(value) if value is not None else None) != expected[key]:\n            raise ValueError(f\'Destination run mismatch: {expected["id"]} field {key}\')\n    for key in (\'start_time\', \'end_time\'):\n        if iso(getattr(actual, key, None)) != iso(expected[key]):\n            raise ValueError(f\'Destination run mismatch: {expected["id"]} field {key}\')\n    metadata = (getattr(actual, \'extra\', None) or {}).get(\'metadata\', {})\n    for key, value in expected[\'extra\'][\'metadata\'].items():\n        if metadata.get(key) != value:\n            raise ValueError(f\'Destination run metadata mismatch: {key}\')\n\n\nclass LangSmithDestination:\n    def __init__(self):\n        from langsmith import Client\n        from langsmith.utils import LangSmithNotFoundError\n        self.client = Client(auto_batch_tracing=False)\n        self.not_found = LangSmithNotFoundError\n        # Workspace identity must be explicit to prevent accidental cross-workspace resume.\n        workspace = os.getenv(\'LANGSMITH_WORKSPACE_ID\')\n        if not workspace:\n            raise ValueError(\'Set LANGSMITH_WORKSPACE_ID explicitly for migration checkpoints.\')\n        self.identity = digest({\'endpoint\': self.client.api_url, \'workspace\': workspace})\n\n    def optional(self, function, **kwargs):\n        try:\n            return function(**kwargs)\n        except self.not_found:\n            return None\n\n    def dataset(self, plan):\n        name = plan[\'dataset_name\']\n        obj = self.optional(self.client.read_dataset, dataset_name=name)\n        provenance = {\'migration_bundle_digest\': plan[\'bundle_digest\']}\n        if obj is None:\n            obj = self.client.create_dataset(dataset_name=name, metadata=provenance,\n                description=\'Imported MLflow dataset; expectations are references.\')\n        metadata = obj.metadata or {}\n        if metadata.get(\'migration_bundle_digest\') != plan[\'bundle_digest\']:\n            raise ValueError(\'Existing dataset does not belong to this exact bundle\')\n        return str(obj.id)\n\n    def example(self, item, dataset_id):\n        obj = self.optional(self.client.read_example, example_id=item[\'id\'])\n        if obj is None:\n            obj = self.client.create_example(example_id=item[\'id\'], dataset_id=dataset_id,\n                inputs=item[\'inputs\'], outputs=item[\'outputs\'], metadata=item[\'metadata\'])\n        if (str(obj.dataset_id) != dataset_id or obj.inputs != item[\'inputs\'] or obj.outputs != item[\'outputs\']\n                or any((obj.metadata or {}).get(k) != v for k, v in item[\'metadata\'].items())):\n            raise ValueError(\'Existing example differs from the plan\')\n        return str(obj.id)\n\n    def project(self, name, item, dataset_id, plan):\n        obj = self.optional(self.client.read_project, project_name=name)\n        metadata = {**item[\'metadata\'], \'migration_bundle_digest\': plan[\'bundle_digest\']}\n        reference = dataset_id if item[\'kind\'] == \'evaluation\' else None\n        if obj is None:\n            obj = self.client.create_project(project_name=name, metadata=metadata,\n                reference_dataset_id=reference, description=\'Imported MLflow snapshot\')\n        if (obj.extra or {}).get(\'metadata\', {}).get(\'migration_bundle_digest\') != plan[\'bundle_digest\']:\n            raise ValueError(\'Existing project belongs to a different bundle\')\n        actual_reference = getattr(obj, \'reference_dataset_id\', None)\n        if (str(actual_reference) if actual_reference else None) != reference:\n            raise ValueError(\'Project reference dataset mismatch\')\n        return str(obj.id)\n\n    def finalize(self, project_id):\n        obj = self.client.read_project(project_id=project_id)\n        if obj.end_time is None:\n            self.client.update_project(project_id=project_id, end_time=datetime.now(timezone.utc),\n                                       name=obj.name, description=obj.description, project_extra=obj.extra)\n        return project_id\n\n    def runs(self, items, perform):\n        """Read/check runs in pages; avoid one GET request per span."""\n        for project_name in sorted({r[\'project_name\'] for r in items}):\n            project = self.client.read_project(project_name=project_name)\n            rows = [r for r in items if r[\'project_name\'] == project_name]\n            existing = {str(r.id): r for r in self.client.list_runs(project_name=project_name)}\n            for start in range(0, len(rows), 50):\n                batch = rows[start:start + 50]\n                def upload(batch=batch):\n                    missing = []\n                    for item in batch:\n                        if item[\'id\'] in existing:\n                            assert_run_matches(existing[item[\'id\']], item)\n                        else:\n                            payload = dict(item)\n                            payload[\'session_id\'] = str(project.id)\n                            missing.append(payload)\n                    if missing:\n                        self.client.batch_ingest_runs(create=missing)\n                    return [r[\'id\'] for r in batch]\n                perform(\'run_batch:\' + project_name + \':\' + str(start), upload)\n\n    def run(self, item):\n        obj = self.optional(self.client.read_run, run_id=item[\'id\'])\n        if obj is None:\n            payload = dict(item)\n            for key in (\'start_time\', \'end_time\'):\n                payload[key] = datetime.fromisoformat(payload[key])\n            self.client.create_run(**payload)\n            # Successful HTTP ingestion can precede read visibility. Check before committing state.\n            for attempt in range(6):\n                obj = self.optional(self.client.read_run, run_id=item[\'id\'])\n                if obj is not None:\n                    break\n                time.sleep(1)\n            if obj is None:\n                raise RuntimeError(\'Run not yet readable; rerun migration with the same state file.\')\n        assert_run_matches(obj, item)\n        project = self.client.read_project(project_name=item[\'project_name\'])\n        if str(obj.session_id) != str(project.id):\n            raise ValueError(\'Run belongs to a different destination project\')\n        return item[\'id\']\n\n    def feedback(self, item):\n        obj = self.optional(self.client.read_feedback, feedback_id=item[\'feedback_id\'])\n        if obj is None:\n            obj = self.client.create_feedback(**item)\n        for key in (\'key\', \'score\', \'value\', \'comment\'):\n            if getattr(obj, key, None) != item[key]:\n                raise ValueError(\'Existing feedback differs from the plan: \' + key)\n        if str(obj.run_id) != item[\'run_id\']:\n            raise ValueError(\'Feedback belongs to a different run\')\n        return str(obj.id)\n\n    def prompt(self, item):\n        from langchain_core.prompts import PromptTemplate\n        obj = self.optional(self.client.pull_prompt, prompt_identifier=item[\'identifier\'])\n        if obj is None:\n            url = self.client.push_prompt(item[\'identifier\'],\n                object=PromptTemplate.from_template(item[\'template\']), is_public=False,\n                description=\'MLflow prompt snapshot; source version \' + item[\'source\'][\'version\'])\n        else:\n            url = None\n        obj = self.client.pull_prompt(item[\'identifier\'])\n        if getattr(obj, \'template\', None) != item[\'template\']:\n            raise ValueError(\'Destination prompt content differs; refusing to overwrite\')\n        return {\'identifier\': item[\'identifier\'], \'url\': url,\n                \'source_version\': item[\'source\'][\'version\'],\n                \'destination_metadata\': {k: v for k, v in (getattr(obj, \'metadata\', None) or {}).items()\n                                         if k in (\'lc_hub_commit_hash\', \'lc_hub_repo\', \'lc_hub_owner\')}}\n\n\n\n\ndef group_migrated_resources(client, output_dir, bundles, prefix):\n    """Attach migrated datasets to the agent represented by the tracing project."""\n    import json\n    from pathlib import Path\n    from stack_common import save_json\n    base = \'/workspaces/current\'\n    value = prefix + \'-\' + bundles[0][\'source\'][\'seed_id\'][:12] + \'-traces\'\n    resources = []\n    for bundle in bundles:\n        state_path = Path(output_dir) / bundle[\'source\'][\'seed_id\'] / \'migration-state.json\'\n        state = json.loads(state_path.read_text())\n        for key, resource_id in state[\'completed\'].items():\n            if key == \'dataset\':\n                resources.append({\'resource_type\': \'dataset\',\n                                  \'resource_id\': resource_id})\n            elif key.startswith(\'prompt:\'):\n                prompt = client.get_prompt(resource_id[\'identifier\'])\n                if prompt is None:\n                    raise ValueError(\'Imported prompt is missing\')\n                resources.append({\'resource_type\': \'prompt\', \'resource_id\': prompt.id})\n    report = attach_agent_resources(client, value, resources)\n    save_json(Path(output_dir) / \'resource-grouping.json\', report)\n    return report\n\n\ndef attach_agent_resources(client, value, resources):\n    """Attach resources to the Agent grouping, not a generic Application tag."""\n    base = \'/workspaces/current\'\n    if not resources:\n        return {\'status\': \'not_needed\', \'resources\': []}\n    def request(method, path, **kwargs):\n        return client.request_with_retries(method, base + path, **kwargs).json()\n    keys = request(\'GET\', \'/tag-keys\')\n    key = next((k for k in keys if k[\'key\'] == \'Agent\'), None)\n    if key is None:\n        key = request(\'POST\', \'/tag-keys\', json={\'key\': \'Agent\'})\n    path = \'/tag-keys/\' + key[\'id\'] + \'/tag-values\'\n    values = request(\'GET\', path)\n    tag = next((v for v in values if v[\'value\'] == value), None)\n    if tag is None:\n        tag = request(\'POST\', path, json={\'value\': value})\n    writes = 0\n    for resource in resources:\n        def tagged():\n            tags = request(\'GET\', \'/tags/resource\', params=resource)\n            return any(v[\'id\'] == tag[\'id\'] for k in tags for v in k.get(\'values\', []))\n        if not tagged():\n            request(\'POST\', \'/taggings\', json={**resource, \'tag_value_id\': tag[\'id\']})\n            writes += 1\n        if not tagged():\n            raise RuntimeError(\'Resource grouping tag failed read-back\')\n    report = {\'status\': \'verified\', \'key\': \'Agent\', \'value\': value,\n              \'taggings_created\': writes, \'resources\': resources}\n    return report\n'}

CODE_DIR = Path(tempfile.mkdtemp(prefix="mlflow-export-"))
for filename, content in MODULES.items():
    (CODE_DIR / filename).parent.mkdir(parents=True, exist_ok=True)
    (CODE_DIR / filename).write_text(content)
    sys.modules.pop(filename.removesuffix(".py").replace("/", "."), None)
sys.path.insert(0, str(CODE_DIR))
from export_mlflow import export
from discover_mlflow import list_experiments, export_experiment
from migration_core import plan_bundle, prepare_plans
from migrate_mlflow import apply_plan, LangSmithDestination
from validate_migration import validate_live
from stack_common import save_json
print("Loaded export and migration scripts.")
if EXPORT_SOURCE and not USE_SETUP_MANIFEST and not SOURCE_EXPERIMENT_ID:
    print(json.dumps(list_experiments(SOURCE_TRACKING_URI or "databricks"), indent=2))
    print("Set SOURCE_EXPERIMENT_ID in the configuration cell, rerun that cell, then continue.")

<a id="export"></a>
## 4. Export existing MLflow resources

Export reads the selected UC experiment, including traces added after setup, current assessments, exact registered prompt versions, native dataset records, and evaluation metrics. It reconciles evaluation traces by run ID and checks case coverage. The SQL warehouse ID is read from the experiment monitoring tag when not already configured. Discovery stops with an error at 10,000 traces rather than silently truncating. Driver-side probe traces are kept separately from evaluated predictions. Missing traces produce `export_errors`; the planner refuses an incomplete export. Retry this cell after transient storage delays.

The bundle contains synthetic input/output payloads. For later production use, review/redact exported content before moving it between systems.

In [ ]:
if EXPORT_SOURCE:
    if USE_SETUP_MANIFEST:
        bundle = export(MANIFEST_PATH, EXPORT_DIR / "bundle.json",
                        tracking_uri=SOURCE_TRACKING_URI, discover_traces=True)
    else:
        if not SOURCE_EXPERIMENT_ID:
            raise ValueError("Choose SOURCE_EXPERIMENT_ID from the listing before exporting.")
        bundle = export_experiment(SOURCE_EXPERIMENT_ID, EXPORT_DIR / "bundle.json",
                                   tracking_uri=SOURCE_TRACKING_URI or "databricks")
else:
    bundle = json.loads(BUNDLE_PATH.read_text())
if bundle.get("export_errors"):
    raise ValueError("Export has gaps; inspect the coverage report before import.")
bundles = bundle["bundles"] if bundle.get("format") == "mlflow-discovery-v1" else [bundle]
print(json.dumps(bundle.get("coverage", {}), indent=2))
print("Exported traces:", sum(len(b["traces"]) for b in bundles))

<a id="preview"></a>
## 5. Preview the migration

Each discovered dataset is kept separate. Trace/assessment and prompt history has its own partition. Review the coverage report and limitations before applying. General discovery archives run metadata; historical evaluation/example links are not inferred. If the destination rejects old timestamps, enable `REPLAY_TIMESTAMPS` below using a fresh export directory and prefix. Replay preserves intra-trace timing and original dates in metadata, but changes displayed dates and gaps between traces. Keep `timestamp-policy.json` for retries. The optional setup-manifest route retains those known mappings.

In [ ]:
REPLAY_TIMESTAMPS = False  # Opt in for historical traces rejected by destination time limits.
plans = prepare_plans(bundles, PREFIX, EXPORT_DIR, replay=REPLAY_TIMESTAMPS)
for plan in plans:
    save_json(EXPORT_DIR / plan["source"]["seed_id"] / "migration-plan.json", plan)
summary = {
    "dataset_rows": sum(len(p["examples"]) for p in plans),
    "traces": sum(len(b["traces"]) for b in bundles),
    "feedback": sum(len(p["feedback"]) for p in plans),
    "prompt_versions": sum(len(p["prompts"]) for p in plans),
    "skipped": [s for p in plans for s in p["skipped"]],
    "limitations": bundle.get("limitations", plans[0]["limitations"] if plans else []),
    "destination_writes": False,
}
save_json(EXPORT_DIR / "export-summary.json", summary)
print(json.dumps(summary, indent=2))

<a id="import"></a>
## 6. Import into LangSmith and validate

This section works locally or in Databricks with the same code. Use an environment with `langsmith`; prompt snapshots also need `langchain-core`. Configure `LANGSMITH_API_KEY`, `LANGSMITH_WORKSPACE_ID`, and optional `LANGSMITH_ENDPOINT` through environment variables or Databricks secret-backed configuration. Never put credential values in cells or saved outputs.

First review the plan. Set `APPLY_MIGRATION=True` to write supported resources and immediately run live read-back validation. Keep the state file in persistent storage for retries. No source credentials are needed here if the bundle is already exported.

In [ ]:
APPLY_MIGRATION = False
INCLUDE_PROMPTS = True
STATE_PATH = EXPORT_DIR / "migration-state.json"

if APPLY_MIGRATION:
    from utils.mlflow_destination import group_migrated_resources
    destination = LangSmithDestination()
    for plan in plans:
        state_path = EXPORT_DIR / plan["source"]["seed_id"] / "migration-state.json"
        state = apply_plan(plan, destination, state_path, include_prompts=INCLUDE_PROMPTS)
        validation = validate_live(plan, destination, state)
        save_json(state_path.parent / "validation-report.json", validation)
        print("Live validation passed:", validation["passed"])
        if not validation["passed"]:
            raise RuntimeError("Inspect validation-report.json before claiming migration completion.")
    print(group_migrated_resources(destination.client, EXPORT_DIR, bundles, PREFIX))
else:
    print("Dry run only. No LangSmith writes performed.")

<a id="continue"></a>
## 7. Send new application traces to LangSmith

Run this opt-in check in either environment after successful migration. It sends a fresh parent/tool trace to a separate development project, demonstrating the same instrumentation you can use in your application. It does not modify imported history.

For the Databricks path, keep your compute and tools there and apply this instrumentation to your existing application. For the local path, run the application locally after adapting any Databricks-only dependencies. Configure destination access in whichever environment executes it.

The companion `providers/mlflow/runbook.ipynb` follows the upstream Phoenix/Langfuse pattern with MLflow-to-LangSmith examples for tracing, datasets, evaluations, and prompts. Online scorer schedules, review workflows, agent deployments, and UC storage configuration are not recreated by the historical import; configure equivalent ongoing workflows separately in LangSmith.

In [ ]:
RUN_LANGSMITH_CHECK = False
DEVELOPMENT_PROJECT = PREFIX + "-development"

if RUN_LANGSMITH_CHECK:
    from langsmith import Client, traceable, tracing_context
    development_client = Client()

    @traceable(name="lookup_order", run_type="tool", client=development_client)
    def lookup_order(order_id):
        return {"order_id": order_id, "eta": "Friday", "synthetic": True}

    @traceable(name="support_agent", run_type="chain", client=development_client)
    def support_agent(order_id):
        order = lookup_order(order_id)
        return {"answer": f"Your order is expected {order['eta']}."}

    with tracing_context(enabled=True, project_name=DEVELOPMENT_PROJECT,
                         client=development_client):
        print(support_agent("DEMO-101"))
    development_client.flush()
    print("Check the new parent/tool trace in LangSmith project:", DEVELOPMENT_PROJECT)
else:
    print("New application tracing check skipped.")

## CLI alternative

From the local migration project, the same stages are available independently:

```sh
python export_mlflow.py --manifest source/manifest.json --output bundle.json --tracking-uri databricks://YOUR_PROFILE
python migrate_mlflow.py --bundle bundle.json --report migration-plan.json
python migrate_mlflow.py --bundle bundle.json --apply --include-prompts --state migration-state.json
python validate_migration.py --bundle bundle.json --live --state migration-state.json --report validation-report.json
```

Export needs MLflow source credentials. Applying and validating an existing bundle need only LangSmith credentials. Offline planning needs neither. Do not run concurrent writers with the same prefix/state. A changed bundle requires a new prefix/state. Partial uploads are not rolled back automatically. See `docs/migration-limitations.md` for the supported mappings and fidelity limits.